# Redes neuronales recurrentes

En el módulo anterior, cubrimos representaciones semánticas ricas del texto. La arquitectura que hemos estado usando captura el significado agregado de las palabras en una oración, pero no toma en cuenta el **orden** de las palabras, porque la operación de agregación que sigue a los embeddings elimina esta información del texto original. Debido a que estos modelos no pueden representar el orden de las palabras, no pueden resolver tareas más complejas o ambiguas, como la generación de texto o la respuesta a preguntas.

Para capturar el significado de una secuencia de texto, usaremos una arquitectura de red neuronal llamada **red neuronal recurrente**, o RNN. Al usar una RNN, pasamos nuestra oración a través de la red un token a la vez, y la red produce algún **estado**, que luego pasamos de nuevo a la red con el siguiente token.

![Imagen que muestra un ejemplo de generación con red neuronal recurrente.](../../../../../translated_images/es/rnn.27f5c29c53d727b5.webp)

Dada la secuencia de entrada de tokens $X_0,\dots,X_n$, la RNN crea una secuencia de bloques de red neuronal, y entrena esta secuencia de extremo a extremo usando retropropagación. Cada bloque de la red toma un par $(X_i,S_i)$ como entrada, y produce $S_{i+1}$ como resultado. El estado final $S_n$ o la salida $Y_n$ va a un clasificador lineal para producir el resultado. Todos los bloques de la red comparten los mismos pesos, y se entrenan de extremo a extremo usando una pasada de retropropagación.

> La figura anterior muestra la red neuronal recurrente en su forma desenrollada (a la izquierda), y en una representación recurrente más compacta (a la derecha). Es importante darse cuenta de que todas las celdas RNN tienen los mismos **pesos compartibles**.

Debido a que los vectores de estado $S_0,\dots,S_n$ se pasan a través de la red, la RNN puede aprender dependencias secuenciales entre palabras. Por ejemplo, cuando la palabra *no* aparece en alguna parte de la secuencia, puede aprender a negar ciertos elementos dentro del vector de estado.

Internamente, cada celda RNN contiene dos matrices de pesos: $W_H$ y $W_I$, y un sesgo $b$. En cada paso de la RNN, dada la entrada $X_i$ y el estado de entrada $S_i$, el estado de salida se calcula como $S_{i+1} = f(W_H\times S_i + W_I\times X_i+b)$, donde $f$ es una función de activación (a menudo $\tanh$).

> Para problemas como la generación de texto (que cubriremos en la siguiente unidad) o la traducción automática, también queremos obtener algún valor de salida en cada paso de la RNN. En este caso, también hay otra matriz $W_O$, y la salida se calcula como $Y_i=f(W_O\times S_i+b_O)$.

Veamos cómo las redes neuronales recurrentes pueden ayudarnos a clasificar nuestro conjunto de datos de noticias.

> Para el entorno sandbox, necesitamos ejecutar la siguiente celda para asegurarnos de que la biblioteca requerida esté instalada y los datos estén precargados. Si estás ejecutando localmente, puedes saltarte la siguiente celda.


In [1]:
import sys
!{sys.executable} -m pip install --quiet tensorflow_datasets==4.4.0
!cd ~ && wget -q -O - https://mslearntensorflowlp.blob.core.windows.net/data/tfds-ag-news.tgz | tar xz

In [2]:
import tensorflow as tf
from tensorflow import keras
import tensorflow_datasets as tfds
import numpy as np

# We are going to be training pretty large models. In order not to face errors, we need
# to set tensorflow option to grow GPU memory allocation when required
physical_devices = tf.config.list_physical_devices('GPU') 
if len(physical_devices)>0:
    tf.config.experimental.set_memory_growth(physical_devices[0], True)

ds_train, ds_test = tfds.load('ag_news_subset').values()

Al entrenar modelos grandes, la asignación de memoria GPU puede convertirse en un problema. También puede ser necesario experimentar con diferentes tamaños de minibatch, de modo que los datos quepan en la memoria de la GPU, pero el entrenamiento sea lo suficientemente rápido. Si está ejecutando este código en su propia máquina con GPU, puede experimentar ajustando el tamaño del minibatch para acelerar el entrenamiento.

> **Nota**: Se sabe que ciertas versiones de los controladores NVidia no liberan la memoria después de entrenar el modelo. Estamos ejecutando varios ejemplos en este cuaderno, y esto podría causar que la memoria se agote en ciertas configuraciones, especialmente si está realizando sus propios experimentos como parte del mismo cuaderno. Si encuentra algunos errores extraños al comenzar a entrenar el modelo, es posible que desee reiniciar el núcleo del cuaderno.


In [3]:
batch_size = 16
embed_size = 64

## Clasificador RNN simple

En el caso de una RNN simple, cada unidad recurrente es una red lineal simple, que recibe un vector de entrada y un vector de estado, y produce un nuevo vector de estado. En Keras, esto puede representarse con la capa `SimpleRNN`.

Aunque podemos pasar tokens codificados en one-hot directamente a la capa RNN, esto no es una buena idea debido a su alta dimensionalidad. Por lo tanto, usaremos una capa de embedding para reducir la dimensionalidad de los vectores de palabras, seguida de una capa RNN, y finalmente un clasificador `Dense`.

> **Nota**: En casos donde la dimensionalidad no es tan alta, por ejemplo cuando se usa tokenización a nivel de caracteres, podría tener sentido pasar tokens codificados en one-hot directamente a la celda RNN.


In [4]:
vocab_size = 20000

vectorizer = keras.layers.experimental.preprocessing.TextVectorization(
    max_tokens=vocab_size,
    input_shape=(1,))

model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, embed_size),
    keras.layers.SimpleRNN(16),
    keras.layers.Dense(4,activation='softmax')
])

model.summary()

Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
text_vectorization (TextVect (None, None)              0         
_________________________________________________________________
embedding (Embedding)        (None, None, 64)          1280000   
_________________________________________________________________
simple_rnn (SimpleRNN)       (None, 16)                1296      
_________________________________________________________________
dense (Dense)                (None, 4)                 68        
Total params: 1,281,364
Trainable params: 1,281,364
Non-trainable params: 0
_________________________________________________________________


> **Nota:** Usamos una capa de embedding no entrenada aquí por simplicidad, pero para obtener mejores resultados podemos usar una capa de embedding preentrenada usando Word2Vec, como se describe en la unidad anterior. Sería un buen ejercicio que adaptes este código para que funcione con embeddings preentrenados.

Ahora entrenemos nuestra RNN. En general, las RNN son bastante difíciles de entrenar, porque una vez que las celdas de la RNN se desenrollan a lo largo de la longitud de la secuencia, el número resultante de capas involucradas en la retropropagación es bastante grande. Por lo tanto, necesitamos seleccionar una tasa de aprendizaje más pequeña y entrenar la red con un conjunto de datos más grande para producir buenos resultados. Esto puede tomar bastante tiempo, por lo que es preferible usar una GPU.

Para acelerar las cosas, solo entrenaremos el modelo RNN con los títulos de noticias, omitiendo la descripción. Puedes intentar entrenar con la descripción y ver si puedes lograr que el modelo se entrene.


In [5]:
def extract_title(x):
    return x['title']

def tupelize_title(x):
    return (extract_title(x),x['label'])

print('Training vectorizer')
vectorizer.adapt(ds_train.take(2000).map(extract_title))

Training vectorizer


In [6]:
model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize_title).batch(batch_size),validation_data=ds_test.map(tupelize_title).batch(batch_size))

7500/7500 [==============================] - 82s 11ms/step - loss: 0.6629 - acc: 0.7623 - val_loss: 0.5559 - val_acc: 0.7995


> **Nota** que la precisión probablemente sea menor aquí, porque solo estamos entrenando con títulos de noticias.


## Revisiting variable sequences 

Remember that the `TextVectorization` layer will automatically pad sequences of variable length in a minibatch with pad tokens. It turns out that those tokens also take part in training, and they can complicate convergence of the model.

There are several approaches we can take to minimize the amount of padding. One of them is to reorder the dataset by sequence length and group all sequences by size. This can be done using the `tf.data.experimental.bucket_by_sequence_length` function (see [documentation](https://www.tensorflow.org/api_docs/python/tf/data/experimental/bucket_by_sequence_length)). 

Another approach is to use **masking**. In Keras, some layers support additional input that shows which tokens should be taken into account when training. To incorporate masking into our model, we can either include a separate `Masking` layer ([docs](https://keras.io/api/layers/core_layers/masking/)), or we can specify the `mask_zero=True` parameter of our `Embedding` layer.

> **Note**: This training will take around 5 minutes to complete one epoch on the whole dataset. Feel free to interrupt training at any time if you run out of patience. What you can also do is limit the amount of data used for training, by adding `.take(...)` clause after `ds_train` and `ds_test` datasets.


In [7]:
def extract_text(x):
    return x['title']+' '+x['description']

def tupelize(x):
    return (extract_text(x),x['label'])

model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size,embed_size,mask_zero=True),
    keras.layers.SimpleRNN(16),
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(batch_size),validation_data=ds_test.map(tupelize).batch(batch_size))

7500/7500 [==============================] - 371s 49ms/step - loss: 0.5401 - acc: 0.8079 - val_loss: 0.3780 - val_acc: 0.8822


Ahora que estamos usando enmascaramiento, podemos entrenar el modelo en todo el conjunto de datos de títulos y descripciones.

> **Nota**: ¿Has notado que hemos estado utilizando un vectorizador entrenado en los títulos de noticias y no en todo el cuerpo del artículo? Potencialmente, esto puede hacer que algunos de los tokens se ignoren, por lo que es mejor reentrenar el vectorizador. Sin embargo, podría tener solo un efecto muy pequeño, así que nos mantendremos con el vectorizador preentrenado anterior por simplicidad.


## LSTM: Memoria a largo plazo y corto plazo

Uno de los principales problemas de las RNN es el **desvanecimiento de gradientes**. Las RNN pueden ser bastante largas y pueden tener dificultades para propagar los gradientes hasta la primera capa de la red durante la retropropagación. Cuando esto sucede, la red no puede aprender relaciones entre tokens distantes. Una forma de evitar este problema es introducir **gestión explícita del estado** mediante el uso de **puertas**. Las dos arquitecturas más comunes que introducen puertas son **memoria a largo plazo y corto plazo** (LSTM) y **unidad de relevo con puertas** (GRU). Aquí cubriremos los LSTM.

![Imagen que muestra un ejemplo de una celda long short term memory](../../../../../lessons/5-NLP/16-RNN/images/long-short-term-memory-cell.svg)

Una red LSTM está organizada de manera similar a una RNN, pero hay dos estados que se pasan de capa en capa: el estado real $c$, y el vector oculto $h$. En cada unidad, el vector oculto $h_{t-1}$ se combina con la entrada $x_t$, y juntos controlan lo que sucede con el estado $c_t$ y la salida $h_{t}$ a través de **puertas**. Cada puerta tiene activación sigmoide (salida en el rango $[0,1]$), que puede considerarse como una máscara bit a bit cuando se multiplica por el vector de estado. Los LSTM tienen las siguientes puertas (de izquierda a derecha en la imagen anterior):
* **puerta de olvido** que determina qué componentes del vector $c_{t-1}$ debemos olvidar y cuáles pasar.
* **puerta de entrada** que determina cuánta información del vector de entrada y del vector oculto anterior debe incorporarse al vector de estado.
* **puerta de salida** que toma el nuevo vector de estado y decide qué componentes se usarán para producir el nuevo vector oculto $h_t$.

Los componentes del estado $c$ pueden considerarse como indicadores que pueden activarse o desactivarse. Por ejemplo, cuando encontramos el nombre *Alice* en la secuencia, suponemos que se refiere a una mujer y activamos el indicador en el estado que dice que tenemos un sustantivo femenino en la oración. Cuando luego encontramos las palabras *and Tom*, activamos el indicador que dice que tenemos un sustantivo plural. Por lo tanto, manipulando el estado podemos hacer un seguimiento de las propiedades gramaticales de la oración.

> **Nota**: Aquí hay un gran recurso para entender los aspectos internos de los LSTM: [Understanding LSTM Networks](https://colah.github.io/posts/2015-08-Understanding-LSTMs/) por Christopher Olah.

Aunque la estructura interna de una celda LSTM puede parecer compleja, Keras oculta esta implementación dentro de la capa `LSTM`, por lo que lo único que debemos hacer en el ejemplo anterior es reemplazar la capa recurrente:


In [8]:
model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, embed_size),
    keras.layers.LSTM(8),
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(8),validation_data=ds_test.map(tupelize).batch(8))

15000/15000 [==============================] - 188s 13ms/step - loss: 0.5692 - acc: 0.7916 - val_loss: 0.3441 - val_acc: 0.8870


> **Nota** que entrenar LSTMs también es bastante lento, y puede que no veas mucho aumento en la precisión al comienzo del entrenamiento. Puede que necesites continuar entrenando por un tiempo para lograr una buena precisión.


## Redes RNN bidireccionales y multicapa

En nuestros ejemplos hasta ahora, las redes recurrentes operan desde el comienzo de una secuencia hasta el final. Esto nos resulta natural porque sigue la misma dirección en la que leemos o escuchamos el habla. Sin embargo, para escenarios que requieren acceso aleatorio a la secuencia de entrada, tiene más sentido ejecutar el cálculo recurrente en ambas direcciones. Las RNN que permiten cálculos en ambas direcciones se llaman redes RNN **bidireccionales**, y se pueden crear envolviendo la capa recurrente con una capa especial `Bidirectonal`.

> **Nota**: La capa `Bidirectional` hace dos copias de la capa que contiene, y establece la propiedad `go_backwards` de una de esas copias a `True`, haciéndola ir en la dirección opuesta a lo largo de la secuencia.

Las redes recurrentes, unidireccionales o bidireccionales, capturan patrones dentro de una secuencia, y los almacenan en vectores de estado o los devuelven como salida. Al igual que con las redes convolucionales, podemos construir otra capa recurrente seguida de la primera para capturar patrones de nivel superior, construidos a partir de patrones de nivel inferior extraídos por la primera capa. Esto nos lleva a la noción de una **RNN multicapa**, que consiste en dos o más redes recurrentes, donde la salida de la capa anterior se pasa a la siguiente capa como entrada.

![Imagen que muestra una RNN de memoria a largo plazo multicapa](../../../../../translated_images/es/multi-layer-lstm.dd975e29bb2a59fe.webp)

*Imagen de [esta maravillosa publicación](https://towardsdatascience.com/from-a-lstm-cell-to-a-multilayer-lstm-network-with-pytorch-2899eb5696f3) de Fernando López.*

Keras facilita la construcción de estas redes, porque solo necesitas agregar más capas recurrentes al modelo. Para todas las capas excepto la última, necesitamos especificar el parámetro `return_sequences=True`, porque necesitamos que la capa devuelva todos los estados intermedios, y no solo el estado final del cálculo recurrente.

Construyamos un LSTM bidireccional de dos capas para nuestro problema de clasificación.

> **Nota** este código nuevamente tarda bastante en completarse, pero nos da la mayor precisión que hemos visto hasta ahora. Así que tal vez valga la pena esperar y ver el resultado.


In [9]:
model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, 128, mask_zero=True),
    keras.layers.Bidirectional(keras.layers.LSTM(64,return_sequences=True)),
    keras.layers.Bidirectional(keras.layers.LSTM(64)),    
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(batch_size),
          validation_data=ds_test.map(tupelize).batch(batch_size))

5045/7500 [===================>..........] - ETA: 2:33 - loss: 0.3709 - acc: 0.8706

## RNNs para otras tareas

Hasta ahora, nos hemos centrado en usar RNNs para clasificar secuencias de texto. Pero pueden manejar muchas más tareas, como la generación de texto y la traducción automática &mdash; consideraremos esas tareas en la próxima unidad.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Descargo de responsabilidad**:
Este documento ha sido traducido utilizando el servicio de traducción automática [Co-op Translator](https://github.com/Azure/co-op-translator). Aunque nos esforzamos por la precisión, tenga en cuenta que las traducciones automatizadas pueden contener errores o inexactitudes. El documento original en su idioma nativo debe considerarse la fuente autorizada. Para información crítica, se recomienda una traducción profesional humana. No somos responsables de cualquier malentendido o interpretación errónea que surja del uso de esta traducción.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
